In [ ]:
import pandas as pd
import os
import subprocess
import shutil
import glob
from dotenv import load_dotenv
import requests
from pathlib import Path
from datetime import datetime
import random

# === LOAD GITHUB TOKEN ===
load_dotenv('All_tokens.env')
GITHUB_TOKEN = os.getenv('GITHUB_TOKEN')
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in All_tokens.env")

HEADERS = {'Authorization': f'token {GITHUB_TOKEN}'}

# === CONFIGURATION ===
csv_path = r"C:\GitHub\Android-Mobile-Apps\8.1-Project_GitHub_URLs.csv"
clone_dir = Path(r"C:\GitHub\AndroidProjects\Cloned repos")
yml_output_dir = Path(r"C:\GitHub\AndroidProjects\Config Files")
commits_dir = Path(r"C:\GitHub\AndroidProjects\Commits")
build_info_dir = Path(r"C:\GitHub\AndroidProjects\BuildInfo")
metadata_csv = Path(r"C:\GitHub\AndroidProjects\8.2-Project_Metadata.csv")

# === CLEAN OLD DATA ===
for path in [clone_dir, yml_output_dir, commits_dir, build_info_dir]:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

# === LOAD AND CLEAN CSV ===
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip().str.lower()
df = df[df['github_url'].notna()]
df['github_url'] = df['github_url'].astype(str).str.strip()
df = df[df['github_url'].str.startswith("https://")]

project_metadata = []
kept_repos = set()

# === PROCESS EACH PROJECT ===
for i, url in enumerate(df['github_url'], 1):
    parts = url.split('/')
    username, project = parts[-2], parts[-1].replace('.git', '')
    repo_name = f"{username}.{project}"
    print(f"\n🔍 [{i}] Processing {repo_name}...")

    repo_path = clone_dir / repo_name
    subprocess.run(['git', 'clone', '--depth', '1', url, str(repo_path)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # --- Extract .yml/.yaml Files ---
    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(('.yml', '.yaml')):
                full_path = Path(root) / file
                rel_path = full_path.relative_to(repo_path)
                safe_name = f"{repo_name}.{str(rel_path).replace(os.sep, '_')}"
                shutil.copy2(full_path, yml_output_dir / safe_name)

    # --- Extract build.gradle & test lines ---
    gradle_output = build_info_dir / f"{repo_name}_build_info.txt"
    with open(gradle_output, 'w', encoding='utf-8') as out_file:
        for gradle_file in glob.glob(str(repo_path / '**/*.gradle*'), recursive=True):
            with open(gradle_file, 'r', encoding='utf-8', errors='ignore') as f:
                lines = f.readlines()
                test_lines = [line for line in lines if 'test' in line.lower()]
                if test_lines:
                    out_file.write(f"\n--- {gradle_file} ---\n")
                    out_file.writelines(test_lines)

    # --- Commit Log ---
    commit_log_path = commits_dir / f"{repo_name}_commits.txt"
    with open(commit_log_path, 'w', encoding='utf-8') as f:
        subprocess.run(['git', 'log', '--pretty=format:%h | %an | %ad | %s'], cwd=repo_path, stdout=f, stderr=subprocess.DEVNULL)

    # --- GitHub API Metadata ---
    api_url = f"https://api.github.com/repos/{username}/{project}"

    try:
        meta_resp = requests.get(api_url, headers=HEADERS)
        pulls_resp = requests.get(f"{api_url}/pulls?state=all", headers=HEADERS)
        contribs_resp = requests.get(f"{api_url}/contributors", headers=HEADERS)

        if meta_resp.ok:
            data = meta_resp.json()
            size = data.get("size", 0)
            forks = data.get("forks_count", 0)
            stars = data.get("stargazers_count", 0)
        else:
            size = forks = stars = 0

        pulls = len(pulls_resp.json()) if pulls_resp.ok else 0
        contributors = len(contribs_resp.json()) if contribs_resp.ok else 0

        last_commit = subprocess.run(
            ['git', 'log', '-1', '--format=%cd'],
            cwd=repo_path,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            encoding='utf-8'
        ).stdout.strip()

        commits_count = subprocess.run(
            ['git', 'rev-list', '--all', '--count'],
            cwd=repo_path,
            stdout=subprocess.PIPE,
            stderr=subprocess.DEVNULL,
            encoding='utf-8'
        ).stdout.strip()

        project_metadata.append({
            "project": repo_name,
            "commits": commits_count,
            "pull_requests": pulls,
            "contributors": contributors,
            "Size": size,
            "Repo_Forks": forks,
            "Repo_Stars": stars,
            "Last_Commit_Date": last_commit
        })

    except Exception as e:
        print(f"⚠️ Metadata fetch failed for {repo_name}: {e}")

    # --- Clean cloned repo unless randomly selected to keep ---
    if len(kept_repos) < 150 and random.random() < 150 / len(df):
        kept_repos.add(repo_name)
    else:
        shutil.rmtree(repo_path, ignore_errors=True)

# === SAVE METADATA ===
pd.DataFrame(project_metadata).to_csv(metadata_csv, index=False)

print(f"\n✅ Analyzed {len(project_metadata)} projects.")
print(f"📁 Metadata saved to: {metadata_csv}")
print(f"📦 Clones kept: {len(kept_repos)}")



📥 Cloning 15puzzle...

📥 Cloning 1Rramp-Android...

📥 Cloning 2021-1-OSSP2-Barcode-8...

📥 Cloning 2022_2_WAP_APP_TEAM1...

📥 Cloning 2048-android...

📥 Cloning 2048-Battles...

📥 Cloning 4pdaClient-plus...

📥 Cloning 4th_Grade_IS-Master-...

📥 Cloning 6502Android...

📥 Cloning 9tique-android...

📥 Cloning a2ln-app...

📥 Cloning aaaaxy...

📥 Cloning aap-juce-frequalizer...

📥 Cloning aat...

📥 Cloning abcore...

📥 Cloning react-accessible-accordion...

📥 Cloning accrescent...

📥 Cloning AcDisplay...

📥 Cloning acestream-engine-android...

📥 Cloning acs-upb-mobile...

📥 Cloning web-activity-time-tracker...

📥 Cloning ActivityDiary...

📥 Cloning ActivityLauncher...

📥 Cloning coreui-free-bootstrap-admin-template...

📥 Cloning ad-silence...

📥 Cloning AdaptiveCards...

📥 Cloning AdAway...

📥 Cloning adb-wifi-setting-manager...

📥 Cloning AdbClipboard...

📥 Cloning admin-portal...

📥 Cloning adopt-me-compose...

📥 Cloning Advanced-Settings-for-Android-Wear...

📥 Cloning AdventCalenderSlim